[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/11-multimodal.ipynb)

# Multimodal Models
**Module 7 — Lesson 11 | Estimated time: 30 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Compute image and text embeddings with CLIP and measure cross-modal similarity
- Perform zero-shot image classification using CLIP
- Answer questions about images with BLIP (Visual QA)
- Understand the LLaVA vision-language model architecture
- Grasp the concept of multi-modal RAG
- Build a simple image search engine using CLIP embeddings

In [ ]:
!pip install -q transformers torch Pillow requests

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import requests
from io import BytesIO
from transformers import (
    CLIPProcessor, CLIPModel,
    BlipProcessor, BlipForQuestionAnswering, BlipForConditionalGeneration,
)

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. CLIP — Contrastive Language-Image Pre-Training

CLIP (OpenAI, 2021) trains an image encoder and a text encoder jointly so that matching image-text pairs are close in embedding space.

**Training:** cosine similarity of matching pairs is maximised; mismatched pairs are pushed apart (contrastive loss).

**Capability:** without any task-specific fine-tuning, CLIP can perform zero-shot classification by comparing image embeddings to text descriptions.

In [ ]:
# Load CLIP base model (86M image encoder + 63M text encoder)
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('CLIP loaded.')
print(f'Image encoder params: {sum(p.numel() for p in clip_model.vision_model.parameters()):,}')
print(f'Text encoder params:  {sum(p.numel() for p in clip_model.text_model.parameters()):,}')

## 2. Image-Text Similarity with CLIP

In [ ]:
def load_image_from_url(url):
    try:
        resp = requests.get(url, timeout=10)
        return Image.open(BytesIO(resp.content)).convert('RGB')
    except Exception:
        # Fallback: create a synthetic image
        return Image.fromarray(np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8))

# Sample images
image_urls = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/320px-Dog_Breeds.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg',
]
image_labels = ['dog1', 'dog2', 'cat']
images = [load_image_from_url(u) for u in image_urls]

texts = [
    'a photo of a dog',
    'a photo of a cat',
    'a photo of a car',
    'a cute puppy playing outside',
]

# Compute similarities
inputs = clip_proc(text=texts, images=images, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    outputs = clip_model(**inputs)
logits = outputs.logits_per_image  # (n_images, n_texts)
probs  = logits.softmax(dim=1).cpu().numpy()

# Heatmap
fig = plt.figure(figsize=(12, 5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2])

# Show images
ax_imgs = fig.add_subplot(gs[0])
for i, (img, lbl) in enumerate(zip(images, image_labels)):
    ax2 = fig.add_axes([0.02 + i*0.12, 0.2, 0.10, 0.6])
    ax2.imshow(img)
    ax2.set_title(lbl, fontsize=8)
    ax2.axis('off')

# Similarity heatmap
ax_heat = fig.add_subplot(gs[1])
im = ax_heat.imshow(probs, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax_heat)
ax_heat.set_xticks(range(len(texts)))
ax_heat.set_xticklabels([t[:30] for t in texts], rotation=30, ha='right', fontsize=8)
ax_heat.set_yticks(range(len(images)))
ax_heat.set_yticklabels(image_labels)
ax_heat.set_title('CLIP Image-Text Similarity')
for i in range(len(images)):
    for j in range(len(texts)):
        ax_heat.text(j, i, f'{probs[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3. Zero-Shot Image Classification with CLIP

To classify an image into N categories, simply compute cosine similarity between the image embedding and N text embeddings of the form `"a photo of a {class}"`.

In [ ]:
def clip_zero_shot(image, class_names, model, processor, device='cpu', template='a photo of a {}'):
    texts = [template.format(c) for c in class_names]
    inputs = processor(text=texts, images=[image], return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).squeeze().cpu().numpy()
    return dict(zip(class_names, probs))

# Test classification
classes = ['dog', 'cat', 'bird', 'car', 'airplane', 'flower', 'laptop', 'pizza']

for img, lbl in zip(images, image_labels):
    scores = clip_zero_shot(img, classes, clip_model, clip_proc, device)
    top3   = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f'Image [{lbl}]:')
    for cls, score in top3:
        bar = '█' * int(score * 40)
        print(f'  {cls:10s} {score:.3f} {bar}')
    print()

## 4. Visual Question Answering with BLIP

In [ ]:
# Load BLIP VQA model (Salesforce)
blip_vqa_proc  = BlipProcessor.from_pretrained('Salesforce/blip-vqa-base')
blip_vqa_model = BlipForQuestionAnswering.from_pretrained(
    'Salesforce/blip-vqa-base', torch_dtype=dtype if device=='cuda' else torch.float32
).to(device)
blip_vqa_model.eval()
print('BLIP VQA loaded.')

def vqa(image, question):
    inputs = blip_vqa_proc(image, question, return_tensors='pt').to(device)
    with torch.no_grad():
        out = blip_vqa_model.generate(**inputs, max_new_tokens=20)
    return blip_vqa_proc.decode(out[0], skip_special_tokens=True)

# Ask questions about images
try:
    dtype = torch.float16 if device == 'cuda' else torch.float32
except NameError:
    dtype = torch.float32

test_img = images[0]   # dog image
questions = [
    'What animal is in the image?',
    'What is the color of the animal?',
    'Is the animal indoors or outdoors?',
    'How many animals are visible?',
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(test_img); axes[0].set_title('Input Image'); axes[0].axis('off')
axes[1].axis('off')
y = 1.0
for q in questions:
    ans = vqa(test_img, q)
    axes[1].text(0, y, f'Q: {q}', fontsize=9, weight='bold', transform=axes[1].transAxes)
    axes[1].text(0, y-0.08, f'A: {ans}', fontsize=9, color='steelblue', transform=axes[1].transAxes)
    y -= 0.22
axes[1].set_title('VQA Results')
plt.tight_layout(); plt.show()

## 5. LLaVA — Vision-Language Model Overview

LLaVA (Large Language and Vision Assistant) connects a visual encoder to a large language model via a projection layer:

```
Image → CLIP Vision Encoder → Linear Projection → LLM (LLaMA/Vicuna/Mistral)
                                                  ↑
                                              Text tokens
```

The projection layer converts visual tokens into the same embedding space as text tokens, allowing the LLM to reason over both modalities.

**LLaVA variants:**
- `liuhaotian/llava-v1.5-7b` — 7B LLaMA-2 backbone (requires ~14GB VRAM)
- `llava-hf/llava-1.5-7b-hf` — HuggingFace-compatible version

Full inference requires an A100 GPU. The code structure is shown below.

In [ ]:
llava_code = '''
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
import torch
from PIL import Image
import requests

# Requires A100 GPU (40GB) or 4-bit quantisation
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-mistral-7b-hf",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    load_in_4bit=True,    # requires bitsandbytes
).to("cuda")

image = Image.open("my_image.jpg")
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is happening in this image? Describe in detail."},
        ],
    },
]
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(images=image, text=prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(output[0], skip_special_tokens=True))
'''
print('LLaVA usage (requires A100 GPU or quantisation):')
print(llava_code)

## 6. Multi-Modal RAG — Concept

Multi-modal RAG retrieves relevant images and text documents before generating a response:

```
Query (text or image)
    ↓
Embed with CLIP
    ↓
Vector search over image + text database
    ↓
Retrieve top-K items (images + captions)
    ↓
Pass to Vision-Language LLM for answer generation
```

## 7. Image Search Engine with CLIP Embeddings

In [ ]:
# Build a mini image search engine
# Step 1: create a database of images with CLIP embeddings

def get_clip_image_embedding(image, model, processor, device):
    inputs = processor(images=[image], return_tensors='pt').to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # L2 normalise

def get_clip_text_embedding(text, model, processor, device):
    inputs = processor(text=[text], return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)

# Database: use our 3 images + synthetic ones
db_images  = images.copy()
db_labels  = image_labels.copy()

# Add some synthetic images with distinct patterns
for color, name in [([255,0,0], 'red_square'), ([0,0,255], 'blue_square'), ([0,200,0], 'green_square')]:
    arr = np.full((224, 224, 3), color, dtype=np.uint8)
    db_images.append(Image.fromarray(arr))
    db_labels.append(name)

# Compute embeddings for all database images
print('Computing database embeddings...')
db_embeddings = []
for img in db_images:
    emb = get_clip_image_embedding(img, clip_model, clip_proc, device)
    db_embeddings.append(emb.cpu())
db_embeddings = torch.cat(db_embeddings, dim=0)  # (N, 512)
print(f'Database: {len(db_images)} images, embedding shape: {db_embeddings.shape}')

In [ ]:
def text_image_search(query_text, db_embeddings, db_images, db_labels, model, processor, device, top_k=3):
    """Search database images by text query using CLIP."""
    query_emb = get_clip_text_embedding(query_text, model, processor, device).cpu()
    similarities = (db_embeddings @ query_emb.T).squeeze()  # cosine similarity
    top_indices  = similarities.argsort(descending=True)[:top_k]
    results = [
        (db_labels[i], db_images[i], similarities[i].item())
        for i in top_indices
    ]
    return results

# Run searches
queries = [
    'a cute animal',
    'a red color',
    'something blue',
    'nature and wildlife',
]

fig, axes = plt.subplots(len(queries), 4, figsize=(14, len(queries)*3))
for row, query in enumerate(queries):
    results = text_image_search(query, db_embeddings, db_images, db_labels,
                                clip_model, clip_proc, device, top_k=3)
    # Query label column
    axes[row][0].text(0.5, 0.5, f'Query:\n"{query}"', ha='center', va='center',
                      fontsize=10, wrap=True, transform=axes[row][0].transAxes)
    axes[row][0].axis('off')
    axes[row][0].set_facecolor('#f0f0f0')
    for col, (label, img, score) in enumerate(results):
        axes[row][col+1].imshow(img)
        axes[row][col+1].set_title(f'{label}\nsim={score:.3f}', fontsize=9)
        axes[row][col+1].axis('off')
plt.suptitle('CLIP Image Search Engine: Text Query → Top-3 Images', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Image-to-image search: find similar images to a query image
def image_image_search(query_image, db_embeddings, db_images, db_labels, model, processor, device, top_k=3):
    query_emb   = get_clip_image_embedding(query_image, model, processor, device).cpu()
    similarities = (db_embeddings @ query_emb.T).squeeze()
    top_indices  = similarities.argsort(descending=True)[:top_k+1]  # +1 to skip self
    results = []
    for i in top_indices:
        if db_images[i] is not query_image:
            results.append((db_labels[i], db_images[i], similarities[i].item()))
        if len(results) == top_k:
            break
    return results

query_img = images[0]  # dog1
results   = image_image_search(query_img, db_embeddings, db_images, db_labels,
                                clip_model, clip_proc, device, top_k=3)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(query_img); axes[0].set_title('Query Image\n(dog1)'); axes[0].axis('off')
for col, (label, img, score) in enumerate(results):
    axes[col+1].imshow(img)
    axes[col+1].set_title(f'{label}\n{score:.3f}')
    axes[col+1].axis('off')
plt.suptitle('Image-to-Image Search with CLIP', fontsize=12)
plt.tight_layout(); plt.show()

## Practice Exercises

**Exercise 1 — CLIP Prompt Engineering**
Zero-shot classification accuracy depends heavily on the text template. Compare `"a photo of a {}"`, `"a picture of a {}"`, `"a {} in the wild"`, and `"a close-up of a {}"` on 10 images from 5 categories. Which template gives the highest accuracy? How does ensembling templates (averaging embeddings) compare?

**Exercise 2 — Image Captioning with BLIP**
Load `Salesforce/blip-image-captioning-base` (`BlipForConditionalGeneration`) and generate captions for 5 different images. Compare unconditional captions (no text prompt) with conditional captions (prefix like `"a photo of"` or `"a satellite image of"`). How does the prefix affect caption style?

**Exercise 3 — Multi-Modal Search Engine**
Extend the image search engine to support both text and image queries: if the query is a string, use the text encoder; if it's a PIL image, use the vision encoder. Add at least 20 images to the database (download from Wikipedia or use torchvision datasets). Implement and evaluate retrieval accuracy using top-5 recall.